# Step 0.5 試運転（docs/v2-light-model-plan.md §4・Colab 無料 T4 用）

**合否には使わない。** 50 問の本番の測定（Step 1）の前に、流れが Colab で最後まで通るかと、決める値（max-model-len・8GB/12GB の GPU_UTIL）を出す。データは架空の社内規程（evaluation/corpus）だけ。トークン不要（使うモデルは全て公開）。ランタイムは **T4 GPU**。

計画の 7 点のうち、このノートでやる事:

1. 50 問 + N の 8 問（権限 2・拒否 2・数値 2・N 2）を `--no-judge` で回答 → 採点役を別に起動して `--judge-from` で採点（計画 1）
2. 採点役の使う前の確認（`evaluation/judge_precheck.py`・正解の分かっている 5 例）（計画 2）
3. `run_eval` を GPU メモリの記録（`nvidia-smi` を 1 秒ごと）の中で別プロセスとして流す（計画 3）
4. 50 問 + N 12 問の入力の最大トークン数を測り、max-model-len を決める（計画 4）
5. 8GB（7,680 MiB）・12GB（11,776 MiB）の予算から GPU_UTIL を出して起動し、記録の最大値が予算に収まるか見る（計画 5）
6. 実行の記録（モデル ID・revision・vLLM の版・dtype・GPU・コミット・データセットの sha256）を各 `run_meta.json` に書く（計画 7）

**やらない事: 計画 6（Gemma 4 E2B を llama-server で起動できるか）。** このノートは vLLM だけ。Gemma は T4 の vLLM では起動しない（#7・2026-10-19 実測）ので、モデルは予備の sarashina2.2-3b。

流れ（「Run all」で全部流れる。止まるのは準備の失敗だけで、起動の失敗・予算超過・採点役の不合格は結果に記録して先へ進む）:

1. 準備（仮想環境・Qdrant・GPU の記録）→ 索引（vLLM の前に流すので GPU の記録に混ざらない）
2. 測定: vLLM を 8192 / 0.85 で起動し、62 問を `LLM_MAX_TOKENS=1` で流して入力トークン数だけ取る → `max-model-len = ceil((最大 + 1024) / 512) × 512`
3. 予算ごと（8GB → 12GB）: GPU が空くのを待つ → GPU_UTIL = (予算 − 常駐) ÷ カード全体 → 起動 → 8 問 + 入力が最長の問 → 記録の最大値が予算を超えたら 1 回だけ GPU_UTIL を下げて起動し直す（2 回目も超えたら超過として記録。それ以上は下げない）
4. 採点: 採点役を 8192 / 0.85 で別に起動 → 採点役の確認 → 各予算の回答を `--judge-from`
5. まとめの表 → サーバーを落とす

読み方:

- まとめの表の `ok` は試運転の確かめ（全問回答・最長の問・OOM なし・KV の追い出しなし・予算内）。`None` は「取れなかった＝未確認」で、通ったことにはしない
- 採点役は **流れを確かめるための代役**（既定は候補と同じモデル。`judge_role=flow-check-only` を記録）。本物の採点役（別系統の 14B 級 4bit）は Step 1 の前に決める。代役が採点役の確認に通らなくても、ここでは記録するだけ
- 結果は `/content/step05/` に残る。最後のセルで zip にする（本文を含むので、リポジトリに入れるのは run_meta.json とまとめだけにする）

In [ ]:
# 依存の衝突を避けるため、vLLM と RAG 側を別の仮想環境に入れる（rag_connection_check.ipynb と同じ）
!pip -q install uv
!git clone -q --depth 1 -b redesign/v2 https://github.com/Maaa2005/local-rag-agent.git /content/repo
!uv venv -q /content/rag-env
# torch は CPU 版を先に入れる（同じ行で CPU の index を足すと uv が他の依存を解けない）
!uv pip install -q --python /content/rag-env/bin/python torch --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /content/rag-env/bin/python "transformers<5" sentence-transformers==3.3.1 \
    qdrant-client==1.12.1 aiosqlite==0.20.0 pydantic-settings==2.6.1 openai==1.57.0 httpx==0.28.1 pyyaml
!uv venv -q /content/vllm-env && uv pip install -q --python /content/vllm-env/bin/python vllm

In [ ]:
# 設定
# T4 では Gemma 4 が vLLM で起動しない（#7）ので予備を使う。設定は bfloat16 で、T4 は bf16 が使えないので float16 で動く
MODEL = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
EXTRA_ARGS = []
# 採点役（流れを確かめるための代役）。既定は候補と同じ。本物の採点役は Step 1 の前に決める
JUDGE_MODEL = MODEL
JUDGE_EXTRA_ARGS = []
MEASURE_LEN, MEASURE_UTIL = 8192, 0.85   # 測定と採点の起動（入力トークン数を測る前なので長めに取る）
RESIDENT_MIB = 0                          # GPU に載せた埋め込み・リランカーの MiB（Step 0.5 は CPU なので 0）
QDRANT_VER = "v1.12.6"                    # qdrant-client 1.12.1 に合わせる
REPO = "/content/repo"
RAG_PY = "/content/rag-env/bin/python"
VLLM_BIN = "/content/vllm-env/bin/vllm"
OUT = "/content/step05"                   # 結果の置き場所（やり直すときはセルを流す前に消すか名前を替える）
PORT = 8001

In [ ]:
import json, os, pathlib, re, signal, subprocess, sys, threading, time, urllib.request

sys.path.insert(0, REPO)   # evaluation.colab.step05 を使う（標準ライブラリだけなので kernel から import できる）
from evaluation.colab import step05 as s5

OUTP = pathlib.Path(OUT)
if OUTP.exists() and any(OUTP.iterdir()):
    raise RuntimeError(f"{OUT} が空でない（前の結果と混ざるので止める）。消すか設定セルの OUT を替える")
OUTP.mkdir(parents=True, exist_ok=True)

def launch(argv, log, env=None):
    # 新しいプロセスグループで起動する（落とすときに killpg を使うため。pid = pgid）
    f = open(log, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)

def wait(url, proc, log, limit):
    # 起動したら True。プロセスが落ちた・時間切れなら、ログの抜き出しを出して False（止めずに記録へ回す）
    t = time.time()
    while time.time() - t < limit:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(url, timeout=3)
            print("ready:", url, f"{time.time() - t:.0f} 秒")
            return True
        except Exception:
            time.sleep(3)
    lines = [l[:400] for l in pathlib.Path(log).read_text(errors="replace").splitlines()]
    hits = [i for i, l in enumerate(lines) if re.search(r"Traceback|ERROR|Error|Exception|Killed", l)]
    print("--- 起動しない:", url, "（全文は", log, "）---")
    print("\n".join(f"{i + 1}: {lines[i][:300]}" for i in hits[:40]) or "エラーらしい行なし")
    print("--- ログの末尾 30 行 ---")
    print("\n".join(lines[-30:]))
    return False

def kill(proc):
    if proc is None:
        return
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait()

def read_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return s5.parse_smi_mib(out)

def fetch(url, timeout=5):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.read().decode("utf-8", errors="replace")
    except Exception:
        return None

# Qdrant
qd = pathlib.Path("/content/qdrant"); (qd / "bin").mkdir(parents=True, exist_ok=True)
!curl -sSL https://github.com/qdrant/qdrant/releases/download/{QDRANT_VER}/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C /content/qdrant/bin
q_env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH=str(qd / "storage"), QDRANT__SERVICE__HTTP_PORT="6333")
qdrant = launch([str(qd / "bin" / "qdrant")], "/content/qdrant.log", q_env)
if not wait("http://127.0.0.1:6333/readyz", qdrant, "/content/qdrant.log", 60):
    raise RuntimeError("Qdrant が起動しない（準備の失敗なのでここで止める）")

# V（GPU メモリ）: カード全体の使用量を 1 秒ごとに記録する（vLLM の確保分・GPU に載せた物を全部含む）
GPU_MEM = []   # (time.time(), MiB)
def _log_gpu_mem():
    while True:
        try:
            GPU_MEM.append((time.time(), read_mib()))
        except Exception:
            pass
        time.sleep(1)
if "GPU_MEM_THREAD" not in globals():
    GPU_MEM_THREAD = threading.Thread(target=_log_gpu_mem, daemon=True)
    GPU_MEM_THREAD.start()

GPU_NAME, CAP, TOTAL_MIB = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.splitlines()[0].split(",")]
TOTAL_MIB = int(TOTAL_MIB)
VLLM_VER = subprocess.run(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                          capture_output=True, text=True).stdout.strip() or "不明"
BASELINE_MIB = read_mib()   # 何も載せていない時の使用量（GPU が空いたかの基準）
REV = s5.hf_revision(MODEL)
JUDGE_REV = s5.hf_revision(JUDGE_MODEL)
print(GPU_NAME, "compute", CAP, "全体", TOTAL_MIB, "MiB / 起動前", BASELINE_MIB, "MiB / vllm", VLLM_VER)
print("revision:", REV, "/ 採点役:", JUDGE_REV)

In [ ]:
# 1. 索引（別プロセス・vLLM の前）。DATA_DIR・QDRANT_URL・QDRANT_COLLECTION は index.json に残り、評価側はそこから作る
WORK = OUTP / "index_work"
COLLECTION = f"step05_{int(time.time())}"
p = subprocess.run([RAG_PY, f"{REPO}/evaluation/colab/corpus_index.py", "--repo", REPO, "--work", str(WORK),
                    "--qdrant-url", "http://127.0.0.1:6333", "--collection", COLLECTION],
                   cwd=REPO, capture_output=True, text=True)
print(p.stdout[-2000:], p.stderr[-3000:])
if p.returncode != 0:
    raise RuntimeError("索引に失敗（準備の失敗なのでここで止める）")
INDEX = json.loads((WORK / "index.json").read_text())
print(INDEX)

In [ ]:
# 2. 測定: 入力トークン数だけ取る（LLM_MAX_TOKENS=1 はこのプロセスだけ）
BASE_URL = f"http://127.0.0.1:{PORT}/v1"
WINDOWS = {}    # 区間名 -> (開始, 終了)。GPU の記録の切り出しに使う

def run_eval(argv, env, name, log):
    t0 = time.time()
    with open(log, "w") as f:
        rc = subprocess.run(argv, cwd=REPO, env=dict(os.environ, **env), stdout=f, stderr=subprocess.STDOUT).returncode
    WINDOWS[name] = (t0, time.time())
    tail = pathlib.Path(log).read_text(errors="replace").splitlines()[-8:]
    print(f"[{name}] rc={rc} {WINDOWS[name][1] - t0:.0f} 秒\n  " + "\n  ".join(tail))
    return rc

MEASURE = {"max_model_len": MEASURE_LEN, "gpu_util": MEASURE_UTIL}
s5.wait_gpu_released(read_mib, BASELINE_MIB)
vllm = launch(s5.vllm_argv(VLLM_BIN, MODEL, PORT, MEASURE_LEN, MEASURE_UTIL, REV.get("sha"), extra=EXTRA_ARGS),
              str(OUTP / "vllm_measure.log"))
if not wait(f"{BASE_URL}/models", vllm, str(OUTP / "vllm_measure.log"), 1200):
    raise RuntimeError("測定用の vLLM が起動しない（max-model-len が決まらないので止める）")
m = re.search(r"dtype=torch\.(\w+)", (OUTP / "vllm_measure.log").read_text(errors="replace"))
DTYPE = m.group(1) if m else "不明"
META_COMMON = {"step": "0.5", "gpu": GPU_NAME, "compute_cap": CAP, "model": MODEL, "model_revision": REV.get("sha"),
               "vllm": VLLM_VER, "dtype": DTYPE, "quant": "なし", "rerank": "off", "note": s5.TRIAL_NOTE}
rc = run_eval(s5.eval_cmd(RAG_PY, OUTP / "measure", meta={**META_COMMON, "purpose": "prompt_tokens",
                                                          "max_model_len": MEASURE_LEN, "gpu_util": MEASURE_UTIL,
                                                          "llm_max_tokens": 1}),
              s5.eval_env(INDEX["env"], BASE_URL, LLM_MAX_TOKENS="1"), "measure", str(OUTP / "measure.log"))
kill(vllm)
STATS = s5.prompt_token_stats(OUTP / "measure" / "raw.jsonl")   # 1 問でも取れなければ止まる
MAX_LEN = s5.needed_max_model_len(STATS["max_prompt_tokens"])
MEASURE.update(STATS, rc=rc, needed_max_model_len=MAX_LEN, dtype=DTYPE)
print("入力の最大", STATS["max_prompt_tokens"], "トークン（", STATS["max_id"], "・", STATS["n"], "問）→ max-model-len", MAX_LEN)
if MAX_LEN > MEASURE_LEN:
    print("注意: 必要な長さが測定の起動（", MEASURE_LEN, "）より長い。測定は通ったので値は使えるが、採点の起動も足りない可能性がある")

In [ ]:
# 3. 予算ごとに起動して 8 問 + 最長の問。予算を超えたら 1 回だけ GPU_UTIL を下げて起動し直す
IDS = s5.trial_ids(STATS["max_id"])
TRIALS = {}   # "8GB" / "8GB-2" など -> 結果

def trial(label, budget, util):
    rec = {"budget": budget, "budget_mib": s5.BUDGETS_MIB[budget], "gpu_util": util, "max_model_len": MAX_LEN}
    s5.wait_gpu_released(read_mib, BASELINE_MIB)
    log = OUTP / f"vllm_{label}.log"
    t_launch = time.time()
    proc = launch(s5.vllm_argv(VLLM_BIN, MODEL, PORT, MAX_LEN, util, REV.get("sha"), extra=EXTRA_ARGS), str(log))
    up = wait(f"{BASE_URL}/models", proc, str(log), 1200)
    WINDOWS[f"{label}:launch"] = (t_launch, time.time())
    if not up:
        kill(proc)
        rec.update(started=False, peak_mib=s5.peak_in_window(GPU_MEM, t_launch, time.time())["peak_mib"])
        TRIALS[label] = rec
        print(f"[{label}] 起動しない（GPU_UTIL {util}）。記録して先へ進む")
        return rec
    log_start = len(log.read_text(errors="replace"))
    before = s5.parse_prometheus_counter(fetch(f"http://127.0.0.1:{PORT}/metrics") or "")
    out_dir = OUTP / f"trial_{label}"
    rc = run_eval(s5.eval_cmd(RAG_PY, out_dir, IDS, meta={**META_COMMON, "budget": budget,
                                                          "budget_mib": s5.BUDGETS_MIB[budget], "gpu_util": util,
                                                          "max_model_len": MAX_LEN}),
                  s5.eval_env(INDEX["env"], BASE_URL), f"{label}:run", str(OUTP / f"trial_{label}.log"))
    metrics_text = fetch(f"http://127.0.0.1:{PORT}/metrics") or ""
    after = s5.parse_prometheus_counter(metrics_text)
    log_scan = s5.scan_vllm_log(log.read_text(errors="replace")[log_start:])
    kill(proc)
    mem = s5.gpu_mem_record(GPU_MEM, {"launch": WINDOWS[f"{label}:launch"], "run": WINDOWS[f"{label}:run"]},
                            s5.BUDGETS_MIB[budget], TOTAL_MIB, BASELINE_MIB)
    checks = s5.trial_checks(out_dir / "raw.jsonl", IDS, STATS["max_id"], mem,
                             s5.counter_delta(before, after), log_scan)
    if (out_dir / "run_meta.json").exists():
        # --judge-from は元の run_meta.json を採点の時点で写すので、採点の前に足す
        s5.attach_to_run_meta(out_dir, "gpu_mem", mem)
        s5.attach_to_run_meta(out_dir, "trial_checks", checks)
        s5.attach_to_run_meta(out_dir, "vllm_log_scan", log_scan)
    rec.update(started=True, rc=rc, out_dir=str(out_dir), peak_mib=mem["peak_mib"], checks=checks,
               preempt_metric_lines=s5.metric_lines(metrics_text)[:10])
    TRIALS[label] = rec
    print(f"[{label}] 最大 {mem['peak_mib']} MiB / 予算 {s5.BUDGETS_MIB[budget]} MiB → ok={checks['ok']}")
    return rec

for budget, budget_mib in s5.BUDGETS_MIB.items():
    util = s5.gpu_util_for_budget(budget_mib, TOTAL_MIB, RESIDENT_MIB)
    rec = trial(budget, budget, util)
    peak = rec.get("peak_mib")
    if peak is not None and peak > budget_mib:
        try:
            new = s5.adjusted_gpu_util(util, peak, budget_mib, TOTAL_MIB)
        except ValueError as exc:   # 超過が大きすぎて下げる余地が無い。超過のまま記録して次の予算へ
            rec["retry"] = f"起動し直さない: {exc}"
            print(f"[{budget}] {rec['retry']}")
            continue
        print(f"[{budget}] 予算超過 {peak - budget_mib} MiB → GPU_UTIL {util} → {new} で 1 回だけ起動し直す")
        trial(f"{budget}-2", budget, new)

In [ ]:
# 4. 採点: 採点役を別に起動 → 採点役の確認 → 各予算の回答を --judge-from（採点役の LLM_MAX_TOKENS は既定の 1024）
JUDGE = {"model": JUDGE_MODEL, "revision": JUDGE_REV.get("sha"), "role": "flow-check-only"}
s5.wait_gpu_released(read_mib, BASELINE_MIB)
judge_log = OUTP / "vllm_judge.log"
judge = launch(s5.vllm_argv(VLLM_BIN, JUDGE_MODEL, PORT, MEASURE_LEN, MEASURE_UTIL, JUDGE_REV.get("sha"),
                            extra=JUDGE_EXTRA_ARGS), str(judge_log))
JUDGE["started"] = wait(f"{BASE_URL}/models", judge, str(judge_log), 1200)
JUDGED = {}
if JUDGE["started"]:
    env = dict(os.environ, **s5.eval_env(INDEX["env"], BASE_URL))
    pre_out = OUTP / "judge_precheck.json"
    p = subprocess.run([RAG_PY, "-m", "evaluation.judge_precheck", "--out", str(pre_out)],
                       cwd=REPO, env=env, capture_output=True, text=True)
    print(p.stdout[-3000:], p.stderr[-3000:])
    # 落ちて JSON が無いときも未判定（judge_precheck の終了コード 0=pass / 1=fail / 3=未判定）
    JUDGE["precheck"] = json.loads(pre_out.read_text())["result"] if pre_out.exists() else "未判定"
    JUDGE["precheck_rc"] = p.returncode
    for label, rec in TRIALS.items():
        raw = pathlib.Path(rec.get("out_dir", "")) / "raw.jsonl"
        if not rec.get("started") or not raw.exists():
            continue
        out_dir = OUTP / f"judged_{label}"
        rc = run_eval(s5.judge_cmd(RAG_PY, raw, out_dir, meta={"step": "0.5", "judge_model": JUDGE_MODEL,
                                                               "judge_revision": JUDGE["revision"],
                                                               "judge_role": JUDGE["role"],
                                                               "judge_precheck": JUDGE["precheck"],
                                                               "note": s5.TRIAL_NOTE}),
                      s5.eval_env(INDEX["env"], BASE_URL), f"judge:{label}", str(OUTP / f"judged_{label}.log"))
        JUDGED[label] = {"rc": rc, "out_dir": str(out_dir)}
else:
    JUDGE["precheck"] = "未判定"
    print("採点役が起動しない。回答は trial_* に残っているので、採点役を替えてこのセルだけ流し直せる")
kill(judge)
print(JUDGE)

In [ ]:
# 5. まとめ（試運転・合否には使わない）
rows = []
for label, rec in TRIALS.items():
    c = rec.get("checks") or {}
    rows.append({"label": label, "gpu_util": rec["gpu_util"], "started": rec["started"], "peak_mib": rec.get("peak_mib"),
                 "budget_mib": rec["budget_mib"], **{k: c.get(k) for k in
                 ("all_answered", "longest_ok", "no_oom", "no_preemption", "within_budget", "ok", "missing")},
                 "judged_rc": JUDGED.get(label, {}).get("rc")})
keys = list(rows[0]) if rows else []
print(" | ".join(keys))
for r in rows:
    print(" | ".join(str(r[k]) for k in keys))
SUMMARY = {"note": s5.TRIAL_NOTE, "gpu": GPU_NAME, "compute_cap": CAP, "total_mib": TOTAL_MIB,
           "baseline_mib": BASELINE_MIB, "vllm": VLLM_VER, "model": MODEL, "model_revision": REV, "dtype": DTYPE,
           "measure": MEASURE, "trial_ids": IDS, "trials": rows, "judge": JUDGE, "judged": JUDGED,
           "skipped": ["計画 6: Gemma 4 E2B の llama-server 起動の確認（このノートは vLLM だけ）"]}
(OUTP / "summary.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=2) + "\n")
print(json.dumps({k: SUMMARY[k] for k in ("measure", "judge")}, ensure_ascii=False, indent=1))
print("→", OUTP / "summary.json")

In [ ]:
# サーバーを落とし、結果を zip にする（回答の本文を含む。リポジトリに入れるのは run_meta.json と summary.json だけ）
kill(qdrant)
!cd /content && zip -qr step05_results.zip step05 -x 'step05/index_work/*'
!ls -la /content/step05_results.zip